# 3D Gaussian splatting

*graphics · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/graphics/gaussian_splatting)

A scene is represented by a few hundred coloured, semi-transparent 3D Gaussians. Each one is projected into the camera with the Jacobian of the perspective map, $\Sigma' = J W \Sigma W^{\mathsf T} J^{\mathsf T}$, the splats are depth-sorted and alpha-composited front to back, $C = \sum_i c_i\alpha_i\prod_{j<i}(1 - \alpha_j)$, and Adam fits every mean, scale, rotation, colour and opacity to photos of a synthetic scene. A held-out view checks that the Gaussians learned the 3D shapes, not just the training pictures.

**Vector form**

$$\Sigma = R\,S\,S^{\mathsf T}R^{\mathsf T},\qquad \Sigma' = J\,W\,\Sigma\,W^{\mathsf T}J^{\mathsf T},\qquad \alpha_i(\mathbf u) = o_i\,e^{-\frac12(\mathbf u - \boldsymbol\mu'_i)^{\mathsf T}\Sigma'^{-1}_i(\mathbf u - \boldsymbol\mu'_i)},\qquad C(\mathbf u) = \sum_i \mathbf c_i\,\alpha_i\prod_{j<i}(1 - \alpha_j),\qquad \min\ \lVert C - C^\ast\rVert_1$$

**Index (tensor) form**

$$\Sigma_{mik} = R_{mij}\,s_{mj}^2\,R_{mkj},\qquad \Sigma'_{vmab} = J_{vmai}W_{vij}\,\Sigma_{mjk}\,W_{vlk}J_{vmbl},\qquad \alpha_{vmp} = o_m\,e^{-\frac12 Q_{vmk}\varphi_{pk}},\qquad C_{vpc} = c_{mc}\,\alpha_{vmp}\textstyle\prod_{n<m}(1 - \alpha_{vnp})$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $R_{mij}$ | `quat_to_rotmat(quat)` | (M, 3, 3) | rotation of Gaussian $m$ from a quaternion, normalised inside, so any 4 numbers are a valid rotation |
| $\Sigma_{mik} = R_{mij}s_{mj}^2R_{mkj}$ | `einsum('mij,mj,mkj->mik', R, s2, R)` | (M, 3, 3) | 3D covariance, $j$ summed; symmetric positive definite for any parameters, $s = e^{\log s}$ |
| $x_{vmi} = W_{vij}\mu_{mj} + t_{vi}$ | `einsum('vij,mj->vmi', W, mean) + t` | (V, M, 3) | every Gaussian centre in every camera frame (x right, y down, z = depth) |
| $\mu'_{vma} = f\,x_{vma}/x_{vm3} + c$ | `project(x_cam, f, c)` | (V, M, 2) | pinhole projection of the centres to pixel coordinates |
| $J_{vmai}$ | `projection_jacobian(x_cam, f)` | (V, M, 2, 3) | $\partial\mu'_a/\partial x_i = \begin{pmatrix} f/z & 0 & -fx/z^2\\ 0 & f/z & -fy/z^2\end{pmatrix}$: the local linearisation of the projection |
| $\Sigma'_{vmab}$ | `einsum('vmaj,mjk,vmbk->vmab', JW, cov3, JW)` | (V, M, 2, 2) | 2D covariance of each splat, $j, k$ summed, with $JW = $ einsum('vmai,vij->vmaj'), plus 0.3 px² low-pass |
| $Q_{vmk}\varphi_{pk}$ | `einsum('vmk,pk->vmp', Q, phi)` | (V, M, P) | the quadratic form $d^{\mathsf T}\Sigma'^{-1}d$ expanded in the 6 monomials $\varphi = (u^2, 2uv, v^2, u, v, 1)$ of the pixel: one matrix product |
| $\alpha_{vmp}$ | `(opacity * exp(-0.5 * q)).clamp(max=0.99)` | (V, M, P) | how much of pixel $p$ splat $m$ covers; $o_m = \sigma(\text{logit})$ |
| $\text{sort}_m\ x_{vm3}$ | `x_cam[..., 2].argsort(dim=1)` | (V, M) | depth order per camera; the small per-Gaussian tensors are reordered before the big $(V, M, P)$ ones exist |
| $T_{vmp} = \prod_{n<m}(1 - \alpha_{vnp})$ | `cumprod(1 - alpha, dim=1), shifted by one` | (V, M, P) | transmittance: light left after the splats in front of $m$; non-increasing along $m$ |
| $C_{vpc} = w_{vmp}c_{mc} + T_{\rm end}b_c$ | `einsum('vmp,vmc->vpc', alpha * T, colour) + …` | (V, P, 3) | front-to-back compositing, $m$ summed; the light that gets through shows the white background $b$ |
| $\lVert C - C^\ast\rVert_1$ | `photometric_loss(pred, gt, "l1")` | () | mean absolute pixel error on one random training view per step; Adam updates all $14M$ numbers |

The ground truth is three spheres and a cube, ray cast analytically with Lambertian shading into 12 cameras on two rings around the scene ($64\times 64$ px); a 13th camera between them, at another height, is held out. The Gaussians start at random positions in a cube, grey and faint, with random orientations. Every Adam step renders one training view through the whole pipeline above and back-propagates the L1 photo error into every mean, log-scale, quaternion, colour and opacity. The tensor lesson is the covariance algebra: $\Sigma = RSS^{\mathsf T}R^{\mathsf T}$ stays a valid (symmetric positive definite) covariance whatever the optimiser does, and the projection is not linear, so each Gaussian is pushed through its local linearisation $J$; the 2D splat $\Sigma' = JW\Sigma W^{\mathsf T}J^{\mathsf T}$ is a chain of einsums. The ellipse panel draws these $1\sigma$ ellipses in the held-out view. Densification, simplified: every 50 steps (for the first 70%) the Gaussians whose centres received the largest gradients are split into two copies at $\boldsymbol\mu \pm R\,S\,\boldsymbol\epsilon$ with axes shrunk by 1.6, reusing nearly transparent Gaussians ($o < 0.05$), so $M$ stays fixed; the paper instead clones and splits using view-space gradients, prunes, resets opacities and grows to millions of Gaussians. Other simplifications versus Kerbl et al. (SIGGRAPH 2023): constant colours instead of spherical harmonics (the scene is Lambertian), a random start instead of a structure-from-motion point cloud, an L1 loss without the D-SSIM term, and a dense rasteriser that evaluates every splat at every pixel (the real one sorts splats into $16\times 16$ tiles on the GPU and skips the far ones), which is why the image is small. The held-out PSNR stays a little below the training PSNR: views never seen constrain the Gaussians less. Try fewer views (the held-out view degrades), the L2 loss, densification off, or more Gaussians and steps (sharper edges).


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/graphics/gaussian_splatting/main.py
"""Main loop: synthetic scene → training images → random Gaussians → Adam on the photometric loss → held-out view."""

import time
from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import psnr, render

SEED = 0
N_SNAP = 30  # checkpoints shown in the animation


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    res, n_views = params["resolution"], params["n_views"]
    m = 60 if fast else params["n_gaussians"]
    iterations = 20 if fast else params["iterations"]
    torch.manual_seed(SEED)

    sc = tensors.scene(dev)
    cams = tensors.training_cameras(n_views, res, dev)
    novel = tensors.novel_camera(n_views, res, dev)
    gt = tensors.render_scene(sc, cams)  # (V, P, 3) the training photographs
    gt_novel = tensors.render_scene(sc, novel)  # (1, P, 3) never used for training
    background = torch.tensor(tensors.BACKGROUND, device=dev)
    print(f"scene: 3 spheres + 1 cube, ray cast into {n_views} training views and 1 held-out view of {res}×{res} px")

    g = tensors.init_gaussians(m, SEED, dev)
    n_params = sum(v.numel() for v in g.values())
    print(f"{m} Gaussians × 14 numbers (μ 3, log s 3, q 4, colour 3, opacity 1) = {n_params:,} parameters")

    t0 = time.perf_counter()
    hist = solve.train(g, cams, gt, novel, gt_novel, background, iterations, params["lr"], params["loss"],
                       params["densify"] == "on", 5 if fast else N_SNAP, SEED)  # fmt: skip
    t_train = time.perf_counter() - t0
    first, last = hist["snaps"][0], hist["snaps"][-1]
    if params["densify"] == "on":
        print(f"densification: {hist['n_split']} splits (faint Gaussians reused as smaller copies of busy ones)")
    print(f"Adam, {iterations} steps × 1 random view, {params['loss'].upper()} loss: "
          f"{hist['loss'][0]:.4f} → {hist['loss'][-1]:.4f}  in {t_train:.1f} s")  # fmt: skip
    print(f"PSNR training views {first['psnr_train']:.1f} → {last['psnr_train']:.1f} dB   ·   "
          f"held-out view {first['psnr_novel']:.1f} → {last['psnr_novel']:.1f} dB")  # fmt: skip

    with torch.no_grad():
        _, splats = render(g, novel, background)
        mean_bg = psnr(background.expand_as(gt_novel), gt_novel).item()
    opaque = (splats["opacity"] > 0.5).sum().item()
    print(f"baseline 'all white' on the held-out view: {mean_bg:.1f} dB   ·   {opaque} of {m} Gaussians end up opaque")
    return plot.animate(gt[0].cpu(), gt_novel[0].cpu(), hist, res, out_dir / "gaussian_splatting.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/graphics/gaussian_splatting/equation.py
r"""Physics → tensor translation.

3D Gaussian splatting represents a scene by M anisotropic Gaussians. Gaussian m has a centre μ_m, a covariance
Σ_m, a colour c_m and an opacity o_m. Indices: v = camera (view), m = Gaussian, p = pixel, i, j, k, l = 3D axes,
a, b = 2D image axes.

    covariance     Σ_{mik}  = R_{mij} s_{mj}² R_{mkj}               j summed   (Σ = R S Sᵀ Rᵀ, always SPD)
    to camera      x_{vmi}  = W_{vij} μ_{mj} + t_{vi}               j summed
    projection     u_{vma}  = f x_{vma} / x_{vm3} + c               perspective divide
    2D covariance  Σ'_{vmab} = J_{vmai} W_{vij} Σ_{mjk} W_{vlk} J_{vmbl}   i, j, k, l summed   (Σ' = J W Σ Wᵀ Jᵀ)
    opacity        α_{vmp}  = o_m exp(-½ d_{vmpa} (Σ'⁻¹)_{vmab} d_{vmpb}),  d = pixel - u    a, b summed
                            = o_m exp(-½ Q_{vmk} φ_{pk})     the same, expanded: k summed over 6 monomials
    compositing    C_{vp}   = Σ_m c_m α_{vmp} T_{vmp},   T_{vmp} = Π_{n before m} (1 - α_{vnp})   m in depth order

J is the 2 × 3 Jacobian of the perspective projection: the projection is not linear, so the Gaussian is mapped
by its local linearisation at the centre (the 'EWA splatting' approximation).
"""

import torch

LOW_PASS = 0.3  # px²: added to Σ' so every splat covers at least about one pixel (anti-aliasing)
ALPHA_MAX = 0.99
NEAR = 0.2  # Gaussians closer to the camera than this are dropped


def quat_to_rotmat(q: torch.Tensor) -> torch.Tensor:
    """Unit quaternion (w, x, y, z) → rotation matrix.   q: (M, 4), normalised here → R: (M, 3, 3)."""
    w, x, y, z = (q / q.norm(dim=-1, keepdim=True)).unbind(-1)
    return torch.stack(
        [
            torch.stack([1 - 2 * (y * y + z * z), 2 * (x * y - w * z), 2 * (x * z + w * y)], -1),
            torch.stack([2 * (x * y + w * z), 1 - 2 * (x * x + z * z), 2 * (y * z - w * x)], -1),
            torch.stack([2 * (x * z - w * y), 2 * (y * z + w * x), 1 - 2 * (x * x + y * y)], -1),
        ],
        -2,
    )


def covariance_3d(quat: torch.Tensor, log_scale: torch.Tensor) -> torch.Tensor:
    """Σ_{mik} = R_{mij} s_{mj}² R_{mkj}   (j summed): rotation × axis lengths², SPD by construction.   → (M, 3, 3)."""
    R = quat_to_rotmat(quat)
    s2 = torch.exp(2 * log_scale)  # (M, 3) squared axis lengths
    return torch.einsum("mij,mj,mkj->mik", R, s2, R)


def to_camera(mean: torch.Tensor, W: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    """x_{vmi} = W_{vij} μ_{mj} + t_{vi}   (j summed): Gaussian centres in every camera frame.   → (V, M, 3)."""
    return torch.einsum("vij,mj->vmi", W, mean) + t[:, None, :]


def project(x_cam: torch.Tensor, f: float, c: float) -> torch.Tensor:
    """Pinhole projection u_a = f x_a / x_3 + c for a = 1, 2.   (..., 3) → (..., 2) pixel coordinates."""
    return f * x_cam[..., :2] / x_cam[..., 2:3] + c


def projection_jacobian(x_cam: torch.Tensor, f: float) -> torch.Tensor:
    """J_{ai} = ∂u_a / ∂x_i of the pinhole projection at x_cam.   (V, M, 3) → (V, M, 2, 3).

    J = [[f/z, 0, -f x/z²], [0, f/z, -f y/z²]]
    """
    x, y, z = x_cam.unbind(-1)
    zero = torch.zeros_like(z)
    return torch.stack(
        [torch.stack([f / z, zero, -f * x / z**2], -1), torch.stack([zero, f / z, -f * y / z**2], -1)], -2
    )


def covariance_2d(cov3: torch.Tensor, W: torch.Tensor, J: torch.Tensor) -> torch.Tensor:
    """Σ'_{vmab} = J_{vmai} W_{vij} Σ_{mjk} W_{vlk} J_{vmbl}   (i, j, k, l summed) + low-pass.   → (V, M, 2, 2)."""
    JW = torch.einsum("vmai,vij->vmaj", J, W)  # (V, M, 2, 3): world → pixel linear map
    cov2 = torch.einsum("vmaj,mjk,vmbk->vmab", JW, cov3, JW)
    return cov2 + LOW_PASS * torch.eye(2, device=cov2.device)


def pixel_features(pix: torch.Tensor, c: float) -> torch.Tensor:
    """φ_{pk} = (u², 2uv, v², u, v, 1) of every pixel, with u, v measured from the image centre c.   → (P, 6)."""
    u, v = (pix - c).unbind(-1)
    return torch.stack([u * u, 2 * u * v, v * v, u, v, torch.ones_like(u)], dim=-1)


def splat_alpha(phi: torch.Tensor, mu2: torch.Tensor, cov2: torch.Tensor, opacity: torch.Tensor, c: float):
    """α_{vmp} = o_{vm} exp(-½ d_a (Σ'⁻¹)_{ab} d_b)   (a, b summed), d = pixel - centre of the splat.

    The quadratic form is a polynomial in the pixel coordinates, so expanding it gives ONE matrix product
    q_{vmp} = Q_{vmk} φ_{pk} (k summed) with the Gaussian's coefficients Q = (A, B, C, -2(Aμ_u + Bμ_v),
    -2(Bμ_u + Cμ_v), Aμ_u² + 2Bμ_uμ_v + Cμ_v²), where Σ'⁻¹ = [[A, B], [B, C]]: much faster than forming d.
    phi (P, 6), mu2 (V, M, 2), cov2 (V, M, 2, 2), opacity (V, M) → (V, M, P), clamped to ALPHA_MAX.
    """
    sxx, sxy, syy = cov2[..., 0, 0], cov2[..., 0, 1], cov2[..., 1, 1]  # (V, M)
    det = sxx * syy - sxy**2
    A, B, C = syy / det, -sxy / det, sxx / det  # the 2 × 2 inverse ('conic'), written out
    mu, mv = (mu2 - c).unbind(-1)
    Q = torch.stack(
        [A, B, C, -2 * (A * mu + B * mv), -2 * (B * mu + C * mv), A * mu**2 + 2 * B * mu * mv + C * mv**2], -1
    )
    q = torch.einsum("vmk,pk->vmp", Q, phi)  # d_a (Σ'⁻¹)_{ab} d_b for every splat and pixel
    return (opacity[..., None] * torch.exp(-0.5 * q.clamp(min=0))).clamp(max=ALPHA_MAX)


def composite(alpha: torch.Tensor, colour: torch.Tensor, background: torch.Tensor):
    """Front-to-back alpha compositing C_{vpc} = Σ_m w_{vmp} c_{vmc} + T_final b_c,  w_{vmp} = α_{vmp} T_{vmp}.

    alpha (V, M, P) and colour (V, M, 3) must already be sorted near → far along m. Returns the image (V, P, 3),
    the weights w (V, M, P) and the transmittance T (V, M, P) in front of each Gaussian.
    """
    through = torch.cumprod(1 - alpha, dim=1)  # light that passes Gaussians 1 … m
    T = torch.cat([torch.ones_like(alpha[:, :1]), through[:, :-1]], dim=1)  # exclusive product: Gaussians before m
    w = alpha * T  # (V, M, P) contribution of each Gaussian to each pixel
    image = torch.einsum("vmp,vmc->vpc", w, colour) + through[:, -1, :, None] * background  # m summed
    return image, w, T


def render(g: dict, cams: dict, background: torch.Tensor) -> tuple[torch.Tensor, dict]:
    """The whole differentiable rasteriser: Gaussians g, cameras cams → images (V, P, 3) and the 2D splats."""
    cov3 = covariance_3d(g["quat"], g["log_scale"])  # (M, 3, 3)
    x_cam = to_camera(g["mean"], cams["W"], cams["t"])  # (V, M, 3)
    visible = x_cam[..., 2] > NEAR  # (V, M)
    x_cam = torch.where(visible[..., None], x_cam, torch.tensor([0.0, 0.0, 1.0], device=x_cam.device))  # safe divide
    mu2 = project(x_cam, cams["f"], cams["c"])  # (V, M, 2)
    J = projection_jacobian(x_cam, cams["f"])  # (V, M, 2, 3)
    cov2 = covariance_2d(cov3, cams["W"], J)  # (V, M, 2, 2)
    opacity = torch.sigmoid(g["opacity_logit"])  # (M,)
    colour = torch.sigmoid(g["colour_logit"])  # (M, 3)

    # depth sort per camera on the small per-Gaussian tensors, before the big (V, M, P) ones are made
    order = x_cam[..., 2].argsort(dim=1)  # (V, M) nearest first
    v = torch.arange(order.shape[0], device=order.device)[:, None]
    phi = pixel_features(cams["pix"], cams["c"])  # (P, 6)
    alpha = splat_alpha(phi, mu2[v, order], cov2[v, order], (opacity * visible)[v, order], cams["c"])  # (V, M, P)
    image, _, _ = composite(alpha, colour[order], background)
    return image, {"mu2": mu2, "cov2": cov2, "visible": visible, "colour": colour, "opacity": opacity}


def photometric_loss(pred: torch.Tensor, target: torch.Tensor, kind: str) -> torch.Tensor:
    """Mean L1 (|C - C*|) or L2 ((C - C*)²) difference over views, pixels and colour channels."""
    diff = pred - target
    return diff.abs().mean() if kind == "l1" else (diff**2).mean()


def psnr(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """Peak signal-to-noise ratio in dB for images in [0, 1]: -10 log₁₀ MSE."""
    return -10 * torch.log10(((pred - target) ** 2).mean())


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/graphics/gaussian_splatting/tensors.py
"""Build the concrete tensors: the synthetic scene, the cameras, the ground-truth images and the initial Gaussians."""

import math

import torch

BACKGROUND = (1.0, 1.0, 1.0)  # white
LIGHT = (0.45, -0.35, 0.82)  # direction towards the light (world frame, z up)
AMBIENT = 0.35
RADIUS = 3.8  # camera distance from the origin
FOV = 34.0  # full field of view in degrees


def scene(device: torch.device) -> dict:
    """Three coloured spheres and a cube rotated about z: the 'real world' that the Gaussians must reproduce."""
    return {
        "centres": torch.tensor([[-0.55, -0.05, -0.15], [0.55, 0.45, -0.25], [0.15, -0.55, 0.35]], device=device),
        "radii": torch.tensor([0.45, 0.35, 0.28], device=device),
        "sphere_rgb": torch.tensor([[0.86, 0.15, 0.15], [0.15, 0.39, 0.92], [0.96, 0.62, 0.04]], device=device),
        "box_centre": torch.tensor([0.35, 0.05, -0.2], device=device),
        "box_half": torch.tensor([0.3, 0.3, 0.3], device=device),
        "box_yaw": 0.5,  # rad
        "box_rgb": torch.tensor([0.09, 0.64, 0.29], device=device),
    }


def look_at(centre: torch.Tensor) -> torch.Tensor:
    """World→camera rotation W (3, 3) for a camera at `centre` looking at the origin.

    Rows of W are the camera axes in world coordinates: x right, y down (image rows), z forward (depth).
    """
    forward = -centre / centre.norm()
    up = torch.tensor([0.0, 0.0, 1.0], device=centre.device)
    right = torch.linalg.cross(forward, up)
    right = right / right.norm()
    down = torch.linalg.cross(forward, right)
    return torch.stack([right, down, forward])


def cameras(azimuths: list[float], elevations: list[float], res: int, device: torch.device) -> dict:
    """V pinhole cameras on a sphere of radius RADIUS, all looking at the origin.

    W (V, 3, 3) and t (V, 3) map a world point to camera coordinates x_c = W x + t; f is the focal length in
    pixels and pix (P, 2) the pixel centres (u, v) of a res × res image, P = res².
    """
    az = torch.tensor(azimuths, device=device)
    el = torch.tensor(elevations, device=device)
    centre = RADIUS * torch.stack([el.cos() * az.cos(), el.cos() * az.sin(), el.sin()], dim=-1)  # (V, 3)
    W = torch.stack([look_at(c) for c in centre])  # (V, 3, 3)
    t = -torch.einsum("vij,vj->vi", W, centre)  # (V, 3) so that the camera centre maps to 0
    f = 0.5 * res / math.tan(math.radians(FOV / 2))
    ij = torch.arange(res, device=device, dtype=torch.float32) + 0.5
    v, u = torch.meshgrid(ij, ij, indexing="ij")
    pix = torch.stack([u.reshape(-1), v.reshape(-1)], dim=-1)  # (P, 2) row-major pixels
    return {"W": W, "t": t, "centre": centre, "f": f, "c": res / 2, "res": res, "pix": pix}


def training_cameras(n_views: int, res: int, device: torch.device) -> dict:
    """n_views cameras evenly spaced in azimuth, alternating between a low (10°) and a high (40°) elevation."""
    az = [2 * math.pi * k / n_views for k in range(n_views)]
    el = [math.radians(10.0 if k % 2 == 0 else 40.0) for k in range(n_views)]
    return cameras(az, el, res, device)


def novel_camera(n_views: int, res: int, device: torch.device) -> dict:
    """A held-out view between the first two training cameras, at an elevation none of them used (25°)."""
    return cameras([math.pi / n_views + 0.1], [math.radians(25.0)], res, device)


def _rays(cams: dict) -> tuple[torch.Tensor, torch.Tensor]:
    """Origins (V, P, 3) and unit directions (V, P, 3) of the ray through every pixel of every camera."""
    d_cam = torch.cat([(cams["pix"] - cams["c"]) / cams["f"], torch.ones_like(cams["pix"][:, :1])], dim=-1)  # (P, 3)
    d = torch.einsum("vji,pj->vpi", cams["W"], d_cam)  # W^T d_cam: back to world axes
    d = d / d.norm(dim=-1, keepdim=True)
    return cams["centre"][:, None, :].expand_as(d), d


def render_scene(sc: dict, cams: dict) -> torch.Tensor:
    """Ground truth by ray casting: nearest hit of spheres and cube, Lambertian shading, white background.

    Returns images (V, P, 3) in [0, 1]. This analytic renderer is only the 'camera'; the Gaussians never see it.
    """
    o, d = _rays(cams)
    V, P, _ = o.shape
    t_best = torch.full((V, P), float("inf"), device=o.device)
    normal = torch.zeros(V, P, 3, device=o.device)
    rgb = torch.zeros(V, P, 3, device=o.device)

    for c, r, col in zip(sc["centres"], sc["radii"], sc["sphere_rgb"]):  # |o + t d - c|² = r²
        oc = o - c
        b = (oc * d).sum(-1)
        disc = b**2 - (oc * oc).sum(-1) + r**2
        t = -b - disc.clamp(min=0).sqrt()
        hit = (disc > 0) & (t > 0) & (t < t_best)
        t_best = torch.where(hit, t, t_best)
        n = (o + t[..., None] * d - c) / r
        normal = torch.where(hit[..., None], n, normal)
        rgb = torch.where(hit[..., None], col, rgb)

    # cube: slab test in its own frame (rotated by yaw about z)
    cy, sy = math.cos(sc["box_yaw"]), math.sin(sc["box_yaw"])
    Rb = torch.tensor([[cy, -sy, 0.0], [sy, cy, 0.0], [0.0, 0.0, 1.0]], device=o.device)  # box → world
    ob = torch.einsum("ji,vpj->vpi", Rb, o - sc["box_centre"])  # R^T (o - c)
    db = torch.einsum("ji,vpj->vpi", Rb, d)
    h = sc["box_half"]
    t1, t2 = (-h - ob) / db, (h - ob) / db
    t_near = torch.minimum(t1, t2).amax(-1)
    t_far = torch.maximum(t1, t2).amin(-1)
    hit = (t_far > t_near) & (t_near > 0) & (t_near < t_best)
    p = (ob + t_near[..., None] * db) / h  # on the unit cube surface
    face = p.abs().argmax(-1, keepdim=True)
    nb = torch.zeros_like(p).scatter(-1, face, p.gather(-1, face).sign())
    t_best = torch.where(hit, t_near, t_best)
    normal = torch.where(hit[..., None], torch.einsum("ij,vpj->vpi", Rb, nb), normal)
    rgb = torch.where(hit[..., None], sc["box_rgb"], rgb)

    light = torch.tensor(LIGHT, device=o.device)
    light = light / light.norm()
    shade = AMBIENT + (1 - AMBIENT) * (normal @ light).clamp(min=0)
    bg = torch.tensor(BACKGROUND, device=o.device)
    return torch.where(torch.isfinite(t_best)[..., None], rgb * shade[..., None], bg)


def init_gaussians(m: int, seed: int, device: torch.device) -> dict:
    """M Gaussians with random centres in the cube [-1, 1]³ (no SfM point cloud here), all trainable:

    mean (M, 3), log_scale (M, 3), quat (M, 4) unnormalised rotation, colour_logit (M, 3), opacity_logit (M,)
    """
    gen = torch.Generator().manual_seed(seed)
    g = {
        "mean": torch.rand(m, 3, generator=gen) * 2 - 1,
        "log_scale": torch.full((m, 3), math.log(0.1)),
        "quat": torch.randn(m, 4, generator=gen),
        "colour_logit": torch.zeros(m, 3),  # sigmoid(0) = 0.5: grey
        "opacity_logit": torch.full((m,), math.log(0.3 / 0.7)),  # sigmoid → 0.3
    }
    return {k: v.to(device).requires_grad_(True) for k, v in g.items()}


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/graphics/gaussian_splatting/solve.py
"""Fit the Gaussians to the training images: Adam on the photometric loss, one random view per step, with a simple
split-style densification that recycles transparent Gaussians."""

import math

import torch

from problems._common.progress import steps

from .equation import photometric_loss, psnr, quat_to_rotmat, render

# Per-parameter learning rates relative to `lr`, in the spirit of the paper (positions move slowly, opacity fast).
LR_SCALE = {"mean": 1.0, "log_scale": 2.0, "quat": 1.0, "colour_logit": 5.0, "opacity_logit": 5.0}
DENSIFY_EVERY = 50  # steps
DEAD = 0.05  # Gaussians fainter than this opacity are recycled
SPLIT = 1.6  # a split Gaussian's axes shrink by this factor (as in the paper)


def select(cams: dict, idx: torch.Tensor) -> dict:
    """The sub-batch of cameras idx (the shared intrinsics and pixel grid are kept)."""
    return {**cams, "W": cams["W"][idx], "t": cams["t"][idx], "centre": cams["centre"][idx]}


def densify(g: dict, opt: torch.optim.Adam, grad_acc: torch.Tensor, max_new: int, gen: torch.Generator) -> int:
    """Split the Gaussians whose centres were pushed hardest (largest accumulated |∂L/∂μ|) into two smaller
    copies (axes ÷ 1.6), reusing nearly transparent Gaussians as the new halves, so the total M stays fixed.

    The two halves sit at μ ± R s ε with ε ~ N(0, 1), i.e. a sample from the parent Gaussian. Returns how many
    Gaussians were split.
    """
    with torch.no_grad():
        opacity = torch.sigmoid(g["opacity_logit"])
        dead = (opacity < DEAD).nonzero().squeeze(1)
        alive = opacity >= DEAD
        k = min(len(dead), max_new, int(alive.sum()))
        if k == 0:
            return 0
        parent = torch.where(alive, grad_acc, -1.0).argsort(descending=True)[:k]
        child = dead[:k]
        R = quat_to_rotmat(g["quat"][parent])  # (k, 3, 3)
        eps = torch.randn(k, 3, generator=gen).to(R.device)
        offset = torch.einsum("kij,kj->ki", R, g["log_scale"][parent].exp() * eps)  # a sample from the parent
        for name in ("log_scale", "quat", "colour_logit", "opacity_logit"):
            g[name][child] = g[name][parent]
        g["mean"][child] = g["mean"][parent] + offset
        g["mean"][parent] -= offset
        g["log_scale"][parent] -= math.log(SPLIT)
        g["log_scale"][child] -= math.log(SPLIT)
        for p in g.values():  # forget Adam's running moments for the rows that changed
            state = opt.state.get(p, {})
            for key in ("exp_avg", "exp_avg_sq"):
                if key in state:
                    state[key][parent] = 0
                    state[key][child] = 0
    return k


def train(g: dict, cams: dict, gt: torch.Tensor, novel: dict, gt_novel: torch.Tensor, background: torch.Tensor,
          iterations: int, lr: float, loss_kind: str, densify_on: bool, n_snap: int, seed: int) -> dict:  # fmt: skip
    """Adam on all Gaussian parameters. gt (V, P, 3) are the training images, gt_novel (1, P, 3) the held-out view.

    Returns the loss per step, the number of splits, and at n_snap checkpoints: the renders of training view 0 and
    of the held-out view, the PSNR on four training views and on the held-out view, and the held-out 2D splats.
    """
    gen = torch.Generator().manual_seed(seed)
    opt = torch.optim.Adam([{"params": [g[k]], "lr": lr * s} for k, s in LR_SCALE.items()])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, iterations, eta_min=lr / 10)
    snap_at = set(torch.linspace(0, iterations, n_snap).round().long().tolist())
    n_views = gt.shape[0]
    monitor = torch.arange(0, n_views, max(1, n_views // 4), device=gt.device)  # training views used for the PSNR
    grad_acc = torch.zeros(g["mean"].shape[0], device=gt.device)
    losses, snaps, n_split = [], [], 0

    def snapshot(it: int):
        with torch.no_grad():
            train_img, _ = render(g, select(cams, monitor), background)  # view 0 and a few others
            novel_img, splats = render(g, novel, background)
        snaps.append(
            {
                "it": it,
                "train": train_img[0].cpu(),
                "novel": novel_img[0].cpu(),
                "psnr_train": psnr(train_img, gt[monitor]).item(),
                "psnr_novel": psnr(novel_img, gt_novel).item(),
                **{k: splats[k][0].cpu() for k in ("mu2", "cov2", "visible")},  # held-out view
                "colour": splats["colour"].cpu(),
                "opacity": splats["opacity"].cpu(),
            }
        )

    for it in steps(iterations, "Adam steps"):
        if it in snap_at:
            snapshot(it)
        idx = torch.randint(0, n_views, (1,), generator=gen).to(gt.device)  # one random training view per step
        pred, _ = render(g, select(cams, idx), background)
        loss = photometric_loss(pred, gt[idx], loss_kind)
        opt.zero_grad()
        loss.backward()
        grad_acc += g["mean"].grad.norm(dim=1)
        opt.step()
        sched.step()
        losses.append(loss.item())
        if densify_on and it % DENSIFY_EVERY == DENSIFY_EVERY - 1 and it < 0.7 * iterations:
            n_split += densify(g, opt, grad_acc, g["mean"].shape[0] // 5, gen)
            grad_acc.zero_()
    snapshot(iterations)
    return {"loss": torch.tensor(losses), "snaps": snaps, "n_split": n_split}


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/graphics/gaussian_splatting/plot.py
"""Animate training: a training view and a held-out view (truth vs render), the splats as ellipses, loss and PSNR."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.collections import EllipseCollection

from problems._common.plotting import save_gif

BLUE, RED, GREY = "#2563eb", "#dc2626", "#9ca3af"


def _img(x: torch.Tensor, res: int) -> np.ndarray:
    return x.reshape(res, res, 3).clamp(0, 1).numpy()


def _ellipses(snap: dict, res: int) -> EllipseCollection:
    """1σ ellipses of the 2D splats Σ' in the held-out view, filled with each Gaussian's colour and opacity."""
    lam, vec = torch.linalg.eigh(snap["cov2"])  # (M, 2), (M, 2, 2) ascending eigenvalues
    keep = snap["visible"] & (snap["mu2"] > -5).all(-1) & (snap["mu2"] < res + 5).all(-1)
    lam, vec, mu = lam[keep], vec[keep], snap["mu2"][keep]
    angle = torch.rad2deg(torch.atan2(vec[:, 1, 1], vec[:, 0, 1]))  # direction of the major axis
    rgba = torch.cat([snap["colour"][keep], (0.15 + 0.6 * snap["opacity"][keep])[:, None]], dim=1)
    return EllipseCollection(
        widths=(2 * lam[:, 1].sqrt()).numpy(), heights=(2 * lam[:, 0].sqrt()).numpy(), angles=angle.numpy(),
        units="xy", offsets=mu.numpy(), facecolors=rgba.numpy(), edgecolors=(0.2, 0.2, 0.2, 0.35), linewidths=0.4,
    )  # fmt: skip


def animate(gt_train: torch.Tensor, gt_novel: torch.Tensor, hist: dict, res: int, path: Path) -> Path:
    snaps, loss = hist["snaps"], hist["loss"].numpy()
    its = np.array([s["it"] for s in snaps])
    p_train = np.array([s["psnr_train"] for s in snaps])
    p_novel = np.array([s["psnr_novel"] for s in snaps])
    gt_n = _img(gt_novel, res)

    fig, axes = plt.subplots(2, 4, figsize=(8.6, 4.8))
    titles = ["training view: truth", "training view: splats", "held-out view: truth", "held-out view: splats",
              "held-out: 1σ ellipses of Σ'", "held-out: |error|"]  # fmt: skip
    ims = []
    for k, ax in enumerate(axes.flat[:6]):
        ax.set_title(titles[k], fontsize=8.5)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
    axes[0, 0].imshow(_img(gt_train, res), extent=(0, res, res, 0))
    ims.append(axes[0, 1].imshow(_img(snaps[0]["train"], res), extent=(0, res, res, 0)))
    axes[0, 2].imshow(gt_n, extent=(0, res, res, 0))
    ims.append(axes[0, 3].imshow(_img(snaps[0]["novel"], res), extent=(0, res, res, 0)))
    ax_el = axes[1, 0]
    ax_el.set_xlim(0, res)
    ax_el.set_ylim(res, 0)
    ax_el.set_aspect("equal")
    err = np.abs(_img(snaps[0]["novel"], res) - gt_n).mean(-1)
    ims.append(axes[1, 1].imshow(err, cmap="Greys", vmin=0, vmax=0.5, extent=(0, res, res, 0)))

    ax_l, ax_p = axes[1, 2], axes[1, 3]
    ax_l.semilogy(np.arange(1, len(loss) + 1), loss, color=GREY, lw=0.6, alpha=0.6)
    (loss_line,) = ax_l.plot([], [], color=BLUE, lw=1.3)
    ax_l.set_xlim(0, len(loss))
    ax_l.set_title("photometric loss", fontsize=8.5)
    ax_l.set_xlabel("Adam step", fontsize=7.5)
    ax_l.tick_params(labelsize=7)
    ax_l.minorticks_off()
    (pt_line,) = ax_p.plot([], [], "o-", color=BLUE, ms=2.5, lw=1.2, label="training views")
    (pn_line,) = ax_p.plot([], [], "s-", color=RED, ms=2.5, lw=1.2, label="held-out view")
    ax_p.set_xlim(0, its[-1])
    ax_p.set_ylim(min(p_train.min(), p_novel.min()) - 1, max(p_train.max(), p_novel.max()) + 2)
    ax_p.set_title("PSNR (dB)", fontsize=8.5)
    ax_p.set_xlabel("Adam step", fontsize=7.5)
    ax_p.tick_params(labelsize=7)
    ax_p.legend(fontsize=6.5, loc="lower right")
    counter = fig.text(0.5, 0.01, "", ha="center", va="bottom", fontsize=9)
    state = {"coll": None}

    def update(k: int):
        s = snaps[k]
        ims[0].set_data(_img(s["train"], res))
        ims[1].set_data(_img(s["novel"], res))
        ims[2].set_data(np.abs(_img(s["novel"], res) - gt_n).mean(-1))
        if state["coll"] is not None:
            state["coll"].remove()
        state["coll"] = _ellipses(s, res)
        state["coll"].set_offset_transform(ax_el.transData)
        ax_el.add_collection(state["coll"])
        upto = max(1, s["it"])
        loss_line.set_data(np.arange(1, upto + 1), loss[:upto])
        pt_line.set_data(its[: k + 1], p_train[: k + 1])
        pn_line.set_data(its[: k + 1], p_novel[: k + 1])
        counter.set_text(
            f"step {s['it']}   ·   PSNR training {s['psnr_train']:.1f} dB   ·   held-out {s['psnr_novel']:.1f} dB"
        )

    fig.tight_layout(rect=(0, 0.04, 1, 1))
    return save_gif(fig, update, list(range(len(snaps))), path, fps=4)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_gaussians': 300,
    'iterations': 300,
    'lr': 0.02,
    'loss': 'l1',
    'densify': 'on',
    'n_views': 12,
    'resolution': 64
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.graphics.gaussian_splatting")]:
    del sys.modules[name]
main = importlib.import_module("problems.graphics.gaussian_splatting.main")
problem_dir = Path("problems/graphics/gaussian_splatting")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
